In [1]:
import os
from dotenv import load_dotenv
import json
import pathlib
import aiohttp
import asyncio
from tqdm.notebook import tqdm
import logging
import time
from typing import List
from datetime import datetime

from pydantic_ai import Agent
from pydantic_ai.providers.openrouter import OpenRouterProvider
from pydantic_ai.models.openai import OpenAIChatModel
from pydantic_ai import StructuredDict

from helpers.constants import OPENALEX_NL_UAS

load_dotenv()

logger = logging.getLogger()
logger.handlers = []
logging.basicConfig(level=logging.INFO, format="%(asctime)s %(levelname)s: %(message)s")

def timed_step(name: str, start_time: float):
    elapsed = time.perf_counter() - start_time
    logging.info(f"[✓] {name} completed in {elapsed:.2f} seconds.")
    return time.perf_counter()

pdf_dir = pathlib.Path("./outputs/downloaded_pdfs")
markdown_output_dir = pathlib.Path("./outputs/markdown_output")
json_output_dir = pathlib.Path("./outputs/json_output")
log_output_dir = pathlib.Path("./outputs/logs")

for d in (pdf_dir, markdown_output_dir, json_output_dir, log_output_dir):
    d.mkdir(parents=True, exist_ok=True)

context_path = pathlib.Path("./helpers/context.md")
with context_path.open("r", encoding="utf-8") as f:
    system_prompt_text = f.read()

with open("./helpers/Schema.json", "r", encoding="utf-8") as f:
    schema = json.load(f)

ProjectPublications = StructuredDict(
    schema,
    name="ProjectPublications",
    description="Extracted project publications data.",
)

openrouter_api_key = os.getenv("OPENROUTER_API_KEY", "")
provider = OpenRouterProvider(api_key=openrouter_api_key)
model = OpenAIChatModel("qwen/qwen3-235b-a22b-2507", provider=provider)
agent = Agent(model, output_type=ProjectPublications)

CHAR_THRESHOLD = 512000  # Skip files larger than 512k characters (~128k tokens)

@agent.system_prompt
def get_system_prompt() -> str:
    return system_prompt_text

In [ ]:
# Fetch OpenAlex Results & Build Queue

BROWSER_HEADERS = {
    "User-Agent": (
        "Mozilla/5.0 (Windows NT 10.0; Win64; x64; rv:144.0) "
        "Gecko/20100101 Firefox/144.0"
    ),
    "Accept": "text/html,application/xhtml+xml,application/xml;q=0.9,*/*;q=0.8",
    "Accept-Language": "en-US,en;q=0.5",
    "Accept-Encoding": "gzip, deflate",
    "Connection": "keep-alive",
    "Upgrade-Insecure-Requests": "1",
}

def pick_pdf_url(work: dict) -> tuple[str | None, str | None]:
    loc = work.get("best_oa_location") or work.get("primary_location") or {}
    pdf_url = loc.get("pdf_url")
    landing = loc.get("landing_page_url")

    if not pdf_url:
        pdf_url = (work.get("open_access") or {}).get("oa_url")

    return pdf_url, landing

async def fetch_all_papers_metadata():
    per_page = 100
    uas_list_str = "|".join(OPENALEX_NL_UAS) 
    
    base_params = f"filter=authorships.institutions.lineage:{uas_list_str},has_content.pdf:true&sort=cited_by_count:desc"
    base_api = f"https://api.openalex.org/works?{base_params}"
    
    all_works = []
    
    async with aiohttp.ClientSession(headers=BROWSER_HEADERS) as session:
        print("Fetching total count...")
        async with session.get(f"{base_api}&per_page=1") as count_response:
            if count_response.status != 200:
                 print(f"Failed to fetch count: HTTP {count_response.status}")
                 return []
            count_data = await count_response.json()
            total_count = (count_data.get("meta") or {}).get("count", 0)
            
        print(f"Total works found: {total_count}")
        if total_count == 0:
            return []

        cursor = "*"
        page_num = 1
        
        pbar = tqdm(total=total_count, desc="Fetching Metadata")
        
        while cursor:
            url = f"{base_api}&per_page={per_page}&cursor={cursor}"
            
            try:
                async with session.get(url) as response:
                    if response.status != 200:
                        print(f"Failed to fetch page {page_num}: HTTP {response.status}")
                        break
                        
                    data = await response.json()
                    results = data.get("results", [])
                    meta = data.get("meta", {})
                    
                    if not results:
                        break
                        
                    all_works.extend(results)
                    pbar.update(len(results))
                    
                    cursor = meta.get("next_cursor")
                    page_num += 1
                    
                    # rate limit
                    await asyncio.sleep(0.1)
                    
            except Exception as e:
                print(f"Error on page {page_num}: {e}")
                break
                
        pbar.close()

    print(f"Collected metadata for {len(all_works)} works.")
    
    queue_file = pathlib.Path("outputs/download_queue.csv")
    queue_file.parent.mkdir(parents=True, exist_ok=True)
    
    csv_rows = []
    
    print("Saving metadata and building queue...")
    for i, work in enumerate(tqdm(all_works, desc="Processing Metadata")):
        pdf_url, referer = pick_pdf_url(work)
        
        if not pdf_url:
            continue
            
        openalex_id = work.get("id")
        if openalex_id:
            slug = openalex_id.replace("https://openalex.org/", "")
        else:
            slug = f"openalex_unknown_{i}"
            
        if not slug.lower().endswith('.pdf'):
            filename = slug + '.pdf'
        else:
            filename = slug
            
        json_slug = filename.replace('.pdf', '')
        json_path = json_output_dir / f"{json_slug}.json"
        
        with json_path.open("w", encoding="utf-8") as f:
            json.dump({"meta": work}, f, indent=2, ensure_ascii=False)
            
        # add to csv queue
        csv_rows.append({
            "openalex_id": slug, 
            "filename": filename,
            "pdf_url": pdf_url
        })

    import csv
    with queue_file.open("w", encoding="utf-8", newline="") as f:
        writer = csv.DictWriter(f, fieldnames=["openalex_id", "filename", "pdf_url"])
        writer.writeheader()
        writer.writerows(csv_rows)
        
    print(f"Queue saved to {queue_file} with {len(csv_rows)} items.")
    return queue_file

queue_csv_path = await fetch_all_papers_metadata()

In [ ]:
# Download PDFs from Queue

import csv

BROWSER_HEADERS = {
    "User-Agent": (
        "Mozilla/5.0 (Windows NT 10.0; Win64; x64; rv:144.0) "
        "Gecko/20100101 Firefox/144.0"
    ),
    "Accept": "text/html,application/xhtml+xml,application/xml;q=0.9,*/*;q=0.8",
    "Accept-Language": "en-US,en;q=0.5",
    "Accept-Encoding": "gzip, deflate",
    "Connection": "keep-alive",
    "Upgrade-Insecure-Requests": "1",
}

async def download_pdf_file(session: aiohttp.ClientSession, url: str, save_path: pathlib.Path) -> tuple[bool, int | None]:
    try:
        async with session.get(url) as response:
            status = response.status
            if status != 200:
                return False, status

            content = await response.read()
            content_type = (response.headers.get("Content-Type") or "").lower()

            if not (content_type.startswith("application/pdf") or content[:4] == b"%PDF"):
                return False, status

            save_path.write_bytes(content)
            return True, status
    except Exception:
        return False, None

async def process_download_queue():
    queue_path = pathlib.Path("outputs/download_queue.csv")
    if not queue_path.exists():
        print("Queue file not found. Please run the Fetch step first.")
        return

    queue_items = []
    with queue_path.open("r", encoding="utf-8") as f:
        reader = csv.DictReader(f)
        for row in reader:
            queue_items.append(row)
            
    total_items = len(queue_items)
    print(f"Loaded {total_items} items from queue.")
    
    print("Specify download range (starts at 1).")
    range_input = input(f"Enter range 'start-end' (e.g. 1-100) or 'all' [Default: all]: ").strip()
    
    start_idx = 0
    end_idx = total_items
    
    if range_input.lower() != "all" and "-" in range_input:
        try:
            parts = range_input.split("-")
            s = int(parts[0])
            e = int(parts[1])
            start_idx = max(0, s - 1)
            end_idx = min(total_items, e)
        except ValueError:
            print("Invalid range format. Downloading all.")
            
    items_to_download = queue_items[start_idx:end_idx]
    print(f"Queueing download for {len(items_to_download)} items (Index {start_idx+1} to {end_idx}).")
    
    failed_log = []
    downloaded_count = 0
    
    async with aiohttp.ClientSession(headers=BROWSER_HEADERS) as session:
        pbar = tqdm(total=len(items_to_download), desc="Downloading")
        
        for item in items_to_download:
            url = item.get("pdf_url")
            filename = item.get("filename")
            openalex_id = item.get("openalex_id")
            
            if not url or not filename:
                pbar.update(1)
                continue
                
            save_path = pdf_dir / filename
            
            # Skip if already exists? Maybe user wants to overwrite or repair.
            # Assuming download if asked.
            
            success, status = await download_pdf_file(session, url, save_path)
            
            if success:
                downloaded_count += 1
            else:
                failed_log.append({
                    "openalex_id": openalex_id,
                    "pdf_url": url,
                    "filename": filename,
                    "status_code": status
                })
                
            pbar.update(1)
            await asyncio.sleep(0.1) # Rate limit
            
        pbar.close()

    print(f"Download complete. {downloaded_count} successful, {len(failed_log)} failed.")
    
    if failed_log:
        ts = datetime.now().strftime("%Y%m%d_%H%M%S")
        fail_csv_path = log_output_dir / f"failed_downloads_{ts}.csv"
        
        with fail_csv_path.open("w", encoding="utf-8", newline="") as f:
            writer = csv.DictWriter(f, fieldnames=["openalex_id", "pdf_url", "filename", "status_code"])
            writer.writeheader()
            writer.writerows(failed_log)
            
        print(f"Failed downloads logged to {fail_csv_path}")

await process_download_queue()

In [ ]:
# OPTIONAL: Retry Failed Downloads

async def retry_failed_downloads():
    print("Available log files:")
    logs = list(log_output_dir.glob("failed_downloads_*.csv"))
    for i, p in enumerate(logs):
        print(f"{i+1}. {p.name}")
        
    retry_file_input = input("Enter path to failed downloads CSV or number from list (or 'n' to skip): ").strip()
    
    csv_path = None
    if retry_file_input.isdigit():
        idx = int(retry_file_input) - 1
        if 0 <= idx < len(logs):
            csv_path = logs[idx]
    elif retry_file_input.lower() != 'n' and retry_file_input:
        p = pathlib.Path(retry_file_input)
        if p.exists():
            csv_path = p
    
    if not csv_path:
        print("Skipping retry.")
        return

    print(f"Retrying from {csv_path}...")
    
    items_to_retry = []
    with csv_path.open("r", encoding="utf-8") as f:
        reader = csv.DictReader(f)
        for row in reader:
            items_to_retry.append(row)
            
    if not items_to_retry:
        print("No items to retry.")
        return

    failed_again = []
    success_count = 0
    
    async with aiohttp.ClientSession(headers=BROWSER_HEADERS) as session:
        pbar = tqdm(total=len(items_to_retry), desc="Retrying")
        
        for item in items_to_retry:
            url = item.get("pdf_url")
            filename = item.get("filename")
            openalex_id = item.get("openalex_id")
            
            # Use logic from previous cell if available, else re-implement simple fetch
            # Assuming download_pdf_file is available from previous cell execution
            save_path = pdf_dir / filename
            
            try:
                # We need to ensure download_pdf_file is defined. 
                # If cells are run in order, it is. 
                # If not, let's include a fallback definition here or rely on user running cells in order.
                # Given the flow, user likely runs them in order.
                success, status = await download_pdf_file(session, url, save_path)
            except NameError:
                # Fallback if function not found
                print("Error: download_pdf_file function not found. Please run the Download cell first.")
                return

            if success:
                success_count += 1
            else:
                 failed_again.append({
                    "openalex_id": openalex_id,
                    "pdf_url": url,
                    "filename": filename,
                    "status_code": status
                })
            
            pbar.update(1)
            await asyncio.sleep(0.1)
            
        pbar.close()
        
    print(f"Retry complete. {success_count} recovered, {len(failed_again)} still failed.")
    
    if failed_again:
        ts = datetime.now().strftime("%Y%m%d_%H%M%S")
        new_log_path = log_output_dir / f"failed_downloads_retry_{ts}.csv"
        with new_log_path.open("w", encoding="utf-8", newline="") as f:
            writer = csv.DictWriter(f, fieldnames=["openalex_id", "pdf_url", "filename", "status_code"])
            writer.writeheader()
            writer.writerows(failed_again)
        print(f"Remaining failures logged to {new_log_path}")
    else:
        print("All retried items successfully downloaded!")

await retry_failed_downloads()

In [ ]:
# Convert PDFs to Markdown for AI input

import concurrent.futures
import pdf_converter 
from tqdm import tqdm


converted_md_paths = []
pdf_targets = [str(p) for p in pdf_dir.glob('**/*.pdf')]

if __name__ == "__main__":
    
    with concurrent.futures.ProcessPoolExecutor() as executor:
        futures = [
            executor.submit(pdf_converter.convert_with_logging, p, markdown_output_dir) 
            for p in pdf_targets
        ]
        
        for future in tqdm(concurrent.futures.as_completed(futures), total=len(pdf_targets), desc='Converting PDFs'):
            try:
                result = future.result()
                if result:
                    converted_md_paths.append(result)
            except Exception as e:
                print(f"Process error: {e}")

    print(f"Converted {len(converted_md_paths)} PDFs to markdown in {markdown_output_dir}")

In [ ]:
# Run AI pipeline

import json
from helpers.functions import get_files_below_threshold
from helpers.agent_runner import run_agent_with_widget

only_missing_agent_field = True # check json files to get only files lacking the 'agent' field?

md_path_objects = get_files_below_threshold(markdown_output_dir, CHAR_THRESHOLD)

if only_missing_agent_field:
    print("Filtering: only including files where corresponding JSON lacks 'agent' field...")
    filtered_list = []
    for md_path in md_path_objects:
        json_path = json_output_dir / f"{md_path.stem}.json"
        
        include = True
        if json_path.exists():
            try:
                with json_path.open("r", encoding="utf-8") as f:
                    data = json.load(f)
                if "agent" in data:
                    include = False
            except Exception as e:
                print(f"Error checking {json_path.name}: {e}")
        
        if include:
            filtered_list.append(md_path)
            
    md_path_objects = filtered_list

md_targets = [str(p) for p in md_path_objects]
total_items = len(md_targets)

if md_targets:
    print(f"Found {total_items} eligible markdown files.")
    
    print("Specify processing range (starts at 1).")
    range_input = input(f"Enter range 'start-end' (e.g. 1-100) or 'all' [Default: all]: ").strip()
    
    start_idx = 0
    end_idx = total_items
    
    if range_input.lower() != "all" and "-" in range_input:
        try:
            parts = range_input.split("-")
            s = int(parts[0])
            e = int(parts[1])
            start_idx = max(0, s - 1)
            end_idx = min(total_items, e)
        except ValueError:
            print("Invalid range format. Processing all.")
            
    targets_slice = md_targets[start_idx:end_idx]
    
    print(f'Running agent on {len(targets_slice)} markdown files (Index {start_idx+1} to {end_idx})...')
    
    # Run with widget display and background logging
    summary = await run_agent_with_widget(
        md_paths=targets_slice,
        agent=agent,
        json_output_dir=json_output_dir,
        schema=schema,
        log_dir=log_output_dir,
        concurrency=5
    )
else:
    print(f'No markdown files found in {markdown_output_dir} below threshold {CHAR_THRESHOLD} (or all filtered out).')

In [ ]:
# Post-process JSON outputs, creating smaller files with only essential data.

import pathlib
import json
import re

minimal_output_dir = pathlib.Path("./outputs/json_minimal_outputs")
minimal_output_dir.mkdir(parents=True, exist_ok=True)

filtered_grant_output_dir = pathlib.Path("./outputs/json_filtered_grant_outputs")
filtered_grant_output_dir.mkdir(parents=True, exist_ok=True)

skipped_count = 0
processed_count = 0
grant_count = 0

for json_path in sorted(json_output_dir.glob("*.json")):
    try:
        with json_path.open("r", encoding="utf-8") as f:
            data = json.load(f)
        
        agent_data = data.get("agent")
        
        if not agent_data:
            skipped_count += 1
            continue
            
        meta_data = data.get("meta", {})
        minimal_meta = {
            "id": meta_data.get("id"),
            "doi": meta_data.get("doi"),
            "title": meta_data.get("title"),
        }
        
        minimal_data = {
            "meta": minimal_meta,
            "agent": agent_data,
        }
        
        minimal_json_path = minimal_output_dir / json_path.name
        with minimal_json_path.open("w", encoding="utf-8") as f:
            json.dump(minimal_data, f, indent=2, ensure_ascii=False)
            
        processed_count += 1
        
        grant_raw = agent_data.get("grantNumber")
        if grant_raw and str(grant_raw).strip():
            # nwo project site heuristic: lowercase, remove spaces, hyphens, dots, non-ascii
            val = str(grant_raw).lower()
            val = re.sub(r'[\s\-\.]|[^\x00-\x7F]', '', val)
            
            agent_data["grantNumberNWOHeuristic"] = val
            
            grant_json_path = filtered_grant_output_dir / json_path.name
            with grant_json_path.open("w", encoding="utf-8") as f:
                json.dump(minimal_data, f, indent=2, ensure_ascii=False)
            grant_count += 1
            
    except Exception as e:
        print(f"Error processing {json_path.name}: {e}")

print(f"Minimal JSON files created in {minimal_output_dir}")
print(f"Processed: {processed_count} files")
print(f"Skipped: {skipped_count} files (missing agent data)")
print(f"Filtered Grant JSON files created in {filtered_grant_output_dir}: {grant_count} files")